## Data Preprocessing for Machine Learning

## Objective

To transform the exploratory dataset (`Featured_Retail_Dataset.csv`) into a mathematically consumable format for the three core machine learning tasks identified in the project scope:

1. **Demand Forecasting** (Regression/Time-Series)
2. **Customer Segmentation** (Clustering)
3. **Anomaly Detection** (Isolation Forests/Statistical)

## Preprocessing Strategy

Based on the project components, we must execute the following transformations:

### 1. Feature Encoding (Text  Math)

**Requirement:** *Demand Forecasting & Segmentation*

**The Problem:** Machine Learning models (like K-Means or Linear Regression) cannot calculate with strings like "Winter" or "Type A".

**The Action:**
 * **One-Hot Encoding:** For nominal categories (`Season`: Winter/Spring...).
 * **Ordinal/Label Encoding:** For ordinal categories (`Type`: A/B/C, `Store_Size_Cat`: Small/Medium/Large).
 * **Binary Encoding:** For booleans (`IsHoliday`: True  1).



### 2. Feature Scaling (Normalization)

**Requirement:** *Customer Segmentation (Clustering)*

**The Problem:** Distance-based algorithms (K-Means) are biased by scale. A variable like `Weekly_Sales` (Range: 0–690k) will completely dominate `CPI` (Range: 126–227) and `Unemployment` (Range: 3–14).

**The Action:** 
 - Apply **StandardScaler** (Z-Score Normalization) to force all numerical features onto the same scale (Mean=0, Std=1).
 
### 3. Time-Series Splitting (Preventing Data Leakage)

**Requirement:** *Demand Forecasting & Time-Based Anomaly Detection*

**The Problem:** In retail forecasting, we cannot use random shuffling (standard `train_test_split`). We cannot train on "Future Data" (2012) to predict "Past Data" (2010).

**The Action:**
 * Sort data chronologically by `Date`.
 * **Train Set:** 2010 – 2011 data.
 * **Test Set:** 2012 data.



### 4. Handling Outliers (Negative Sales)

* **Requirement:** *Anomaly Detection & Forecasting*
* **The Problem:** We identified negative sales (returns). While useful for anomaly detection, they can confuse forecasting models trying to predict consumer *demand*.
* **The Action:** Create a "Clean" version for forecasting where negative values are clipped to 0.


In [1]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer

#### Data Ingestion & Standardization

In [2]:
def ingest_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df

def standardize_columns(df: pd.DataFrame) ->pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df 

In [10]:
# 1. LOAD DATA
path = Path('../data/preprocessed_for_analysis/Featured_Retail_Dataset.csv')
df = ingest_data(path)
df = standardize_columns(df)

In [11]:
df

,store,dept,date,weekly_sales,year,month,season,isholiday,type,size,...,fuel_price,cpi,unemployment,markdown1,markdown2,markdown3,markdown4,markdown5,total_markdown,has_promo
0,1,1,2010-02-05,24924.50,2010,2,Winter,False,A,151315,...,2.572,211.09636,8.106,0.00,0.00,0.00,0.00,0.00,0.00,False
1,1,1,2010-02-12,46039.49,2010,2,Winter,True,A,151315,...,2.548,211.24217,8.106,0.00,0.00,0.00,0.00,0.00,0.00,False
2,1,1,2010-02-19,41595.55,2010,2,Winter,False,A,151315,...,2.514,211.28914,8.106,0.00,0.00,0.00,0.00,0.00,0.00,False
3,1,1,2010-02-26,19403.54,2010,2,Winter,False,A,151315,...,2.561,211.31964,8.106,0.00,0.00,0.00,0.00,0.00,0.00,False
4,1,1,2010-03-05,21827.90,2010,3,Spring,False,A,151315,...,2.625,211.35014,8.106,0.00,0.00,0.00,0.00,0.00,0.00,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,98,2012-09-28,508.37,2012,9,Fall,False,B,118221,...,3.997,192.01357,8.684,4556.61,20.64,1.50,1601.01,3288.25,9468.01,True
421566,45,98,2012-10-05,628.10,2012,10,Fall,False,B,118221,...,3.985,192.17041,8.667,5046.74,0.00,18.82,2253.43,2340.01,9659.00,True
421567,45,98,2012-10-12,1061.02,2012,10,Fall,False,B,118221,...,4.000,192.32727,8.667,1956.28,0.00,7.89,599.32,3990.54,6554.03,True
421568,45,98,2012-10-19,760.01,2012,10,Fall,False,B,118221,...,3.969,192.33086,8.667,2004.02,0.00,3.18,437.73,1537.49,3982.42,True


## Preprocessing Pipeline

In [ ]:

# 1. LOAD DATA
path = Path('../data/preprocessed_for_analysis/Featured_Retail_Dataset.csv')
df = ingest_data(path)
df = standardize_columns(df)
df['date'] = pd.to_datetime(df['date'])

# Handle Negative Sales (Target Cleaning)
# We do this in Pandas because sklearn pipelines transform X (Features), not usually y (Target)
df['weekly_sales'] = df['weekly_sales'].clip(lower=0)

# Sort by Date (Critical for Time Series)
df = df.sort_values(by=['date', 'store', 'dept']).reset_index(drop=True)

# 2. DEFINE FEATURE GROUPS
# Added 'markdown' columns to numeric (missed in previous step)
numeric_features = [
    'temperature', 'fuel_price', 'cpi', 'unemployment', 'size', 
    'markdown1', 'markdown2', 'markdown3', 'markdown4', 'markdown5', 'total_markdown'
]

# Added 'temp_range' to Ordinal because Cold < Mild < Hot
ordinal_features = ['type', 'store_size_cat', 'temp_range'] 

# Nominal (No order)
nominal_features = ['season']

# Binary (Already 0/1 or True/False)
binary_features  = ['isholiday', 'has_promo']

# 3. DEFINE THE TRANSFORMERS (The Mapping Logic)
# We must provide the lists in the EXACT order of 'ordinal_features' above:
# 1. Type: C < B < A
# 2. Size: Small < Medium < Large < Super-Large
# 3. Temp: Cold < Mild < Hot  <-- NEW ADDITION
type_order = ['C', 'B', 'A']
size_order = ['Small', 'Medium', 'Large', 'Super Large']
temp_order = ['Cold', 'Mild', 'Hot']

# Pass the lists in the same order as the column names
ordinal_transformer = OrdinalEncoder(categories=[type_order, size_order, temp_order])

# 4. BUILD THE MASTER PROCESSOR
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('nom', OneHotEncoder(drop='first', sparse_output=False), nominal_features),
        ('bin', 'passthrough', binary_features)
    ],
    verbose_feature_names_out=False,
    remainder='drop' # Drop columns not listed (like 'Date')
)

# At the very end of the pipeline, the transformer runs np.hstack() (Horizontal Stack) to glue Belt 1, 2, 3, and 4 together.
# It looks at the data types: [Float, Float, Float, Boolean]
# And converts them into binary (0, 1 integers)

# 5. SPLIT (Time-Based)
train_mask = (df['date'].dt.year <= 2011)
test_mask = (df['date'].dt.year == 2012)

X = df.drop(columns=['weekly_sales', 'date'])
y = df['weekly_sales']

X_train_raw = X.loc[train_mask]
y_train = y.loc[train_mask]
X_test_raw = X.loc[test_mask]
y_test = y.loc[test_mask]

# 6. EXECUTE PIPELINE
# Fit on Train
X_train_processed = preprocessor.fit_transform(X_train_raw)
# Transform Test
X_test_processed = preprocessor.transform(X_test_raw)

# 7. CONVERT TO DATAFRAME
feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train_raw.index)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test_raw.index)

# 8. VERIFY
print("New Columns Check:")
print(X_train_df.columns.tolist())
print("\nFirst 5 rows:")
print(X_train_df.head())

# Save
X_train_df.to_csv('../data/preprocessed_for_models/X_train_processed.csv', index=False)
y_train.to_csv('../data/preprocessed_for_models/y_train.csv', index=False)
X_test_df.to_csv('../data/preprocessed_for_models/X_test_processed.csv', index=False)
y_test.to_csv('../data/preprocessed_for_models/y_test.csv', index=False)

New Columns Check:
['temperature', 'fuel_price', 'cpi', 'unemployment', 'size', 'markdown1', 'markdown2', 'markdown3', 'markdown4', 'markdown5', 'total_markdown', 'type', 'store_size_cat', 'temp_range', 'season_Spring', 'season_Summer', 'season_Winter', 'isholiday', 'has_promo']

First 5 rows:
   temperature  fuel_price       cpi  unemployment      size  markdown1  \
0     -0.87668   -1.452341  1.080658     -0.067814  0.236801  -0.188888   
1     -0.87668   -1.452341  1.080658     -0.067814  0.236801  -0.188888   
2     -0.87668   -1.452341  1.080658     -0.067814  0.236801  -0.188888   
3     -0.87668   -1.452341  1.080658     -0.067814  0.236801  -0.188888   
4     -0.87668   -1.452341  1.080658     -0.067814  0.236801  -0.188888   

   markdown2  markdown3  markdown4  markdown5  total_markdown  type  \
0  -0.103036  -0.098191  -0.157526  -0.193251       -0.207337   2.0   
1  -0.103036  -0.098191  -0.157526  -0.193251       -0.207337   2.0   
2  -0.103036  -0.098191  -0.157526  -0.19

In [9]:
X_train_df.head(5)
# y_train
# X_test_df
# y_test

,temperature,fuel_price,cpi,unemployment,size,markdown1,markdown2,markdown3,markdown4,markdown5,total_markdown,type,store_size_cat,temp_range,season_Spring,season_Summer,season_Winter,isholiday,has_promo
0,-0.87668,-1.452341,1.080658,-0.067814,0.236801,-0.188888,-0.103036,-0.098191,-0.157526,-0.193251,-0.207337,2.0,2.0,1.0,0.0,0.0,1.0,0.0,0.0
1,-0.87668,-1.452341,1.080658,-0.067814,0.236801,-0.188888,-0.103036,-0.098191,-0.157526,-0.193251,-0.207337,2.0,2.0,1.0,0.0,0.0,1.0,0.0,0.0
2,-0.87668,-1.452341,1.080658,-0.067814,0.236801,-0.188888,-0.103036,-0.098191,-0.157526,-0.193251,-0.207337,2.0,2.0,1.0,0.0,0.0,1.0,0.0,0.0
3,-0.87668,-1.452341,1.080658,-0.067814,0.236801,-0.188888,-0.103036,-0.098191,-0.157526,-0.193251,-0.207337,2.0,2.0,1.0,0.0,0.0,1.0,0.0,0.0
4,-0.87668,-1.452341,1.080658,-0.067814,0.236801,-0.188888,-0.103036,-0.098191,-0.157526,-0.193251,-0.207337,2.0,2.0,1.0,0.0,0.0,1.0,0.0,0.0
